In [ ]:
!pip install -q strands-decider


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.7/147.7 kB 4.6 MB/s eta 0:00:00


In [ ]:
!strands-decider ask StrandsAgents/strands-decider-2B-hobson-v19 --device cuda \
  --state "Q: Who directed Jaws? Draft answer: Steven Spielberg." \
  --noul "Is the draft answer correct and fully supported?"

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 37 files:   0% 0/37 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Reconstructing (incomplete total...):   0% 0.00/13.1k [00:00<?, ?B/s]         

Fetching 37 files:   3% 1/37 [00:00<00:04,  7.54it/s]
Reconstructing (incomplete total...):  89% 13.1k/14.6k [00:00<00:00, 99.7kB/s]
Reconstructing (incomplete total...):  91% 14.6k/16.1k [00:00<00:00, 99.7kB/s]
Reconstructing (incomplete total...):  83% 16.1k/19.4k [00:00<00:00, 99.7kB/s]
Reconstructing (incomplete total...):  74% 19.4k/26.2k [00:00<00:00, 99.7kB/s]
Reconstructing (incomplete total...):  57% 19.4k/34.0k [00:00<00:00, 99.7kB/s]
Reconstructing (incomplete total...):  78% 34.0k/43.8k [00:00<00:00, 99.7kB/s]
Reconstructing (incomplete total...):  74% 34.0k/45.7k [00:00<00:00, 99.7kB/s]
Reconstructing (incomplete total...):  95% 45.7k/

In [ ]:
!pip uninstall -y torchao


Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
!strands-decider ask StrandsAgents/strands-decider-2B-hobson-v19 --device cuda \
  --state "Q: Who directed Jaws? Draft answer: Steven Spielberg." \
  --noul "Is the draft answer correct and fully supported?"


In [ ]:
import subprocess, time
srv = subprocess.Popen(["strands-decider", "serve",
    "StrandsAgents/strands-decider-2B-hobson-v19", "--port", "8000"])
time.sleep(60)


In [ ]:
import requests
def strands_critic(question, context, draft):
    state = f"Question: {question}\nEvidence: {context}\nDraft answer: {draft}"
    r = requests.post("http://localhost:8000/v1/systemone", json={
        "state": state,
        "questions": {"sufficient": {"type": "noul",
            "instructions": "Is the draft answer correct and fully supported by the evidence?"}}})
    return r.json()

print(strands_critic("Who directed Jaws?", "Jaws (1975) was directed by Steven Spielberg.", "Steven Spielberg"))

{'model': 'strands-decider-2B-hobson-v19', 'answers': {'sufficient': {'type': 'noul', 'noul': 0.8902}}, 'usage': {'input_tokens': 111, 'output_tokens': 1}, 'latency_ms': 913.06}


In [ ]:
!pip install -q datasets
from datasets import load_dataset
import pandas as pd, random

ds = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation").select(range(50))

def ctx(ex):
    return " ".join(" ".join(s) for s in ex["context"]["sentences"])[:3000]

def score(ex, ans):
    a = strands_critic(ex["question"], ctx(ex), ans)["answers"]["sufficient"]
    return a.get("noul", a.get("value"))

rows = []
for i, ex in enumerate(ds):
    wrong = ds[(i + 7) % len(ds)]["answer"]
    rows.append({"id": ex["id"], "label": 1, "score": score(ex, ex["answer"])})
    rows.append({"id": ex["id"], "label": 0, "score": score(ex, wrong)})

df = pd.DataFrame(rows)
df.to_csv("strands_hotpot50.csv", index=False)

from sklearn.metrics import roc_auc_score
acc = ((df.score > 0.5) == (df.label == 1)).mean()
print("n =", len(df), "| accuracy@0.5 =", round(acc, 3), "| AUROC =", round(roc_auc_score(df.label, df.score), 3))
print(df.groupby("label").score.mean())


README.md:   0%|          | 0.00/9.52k [00:00<?, ?B/s]

distractor/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  166MB            

distractor/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

distractor/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  166MB            

distractor/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

distractor/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 27.5MB            

distractor/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

n = 100 | accuracy@0.5 = 0.7 | AUROC = 0.881
label
0    0.208918
1    0.456260
Name: score, dtype: float64


In [ ]:
import pandas as pd
from sklearn.metrics import roc_auc_score
df = pd.read_csv("jev_drafts.csv")

if df["context"].isna().all():
    from datasets import load_dataset
    hq = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation")
    by_q = {ex["question"]: " ".join(" ".join(s) for s in ex["context"]["sentences"]) for ex in hq}
    df["context"] = df["question"].map(by_q).fillna("")

def score(r):
    a = strands_critic(r.question, str(r.context)[:3000], str(r.draft))["answers"]["sufficient"]
    return a.get("noul", a.get("value"))

df["strands"] = [score(r) for r in df.itertuples()]
df.to_csv("jev_vs_strands.csv", index=False)

for c in ["jev", "strands"]:
    approve = df[c] >= 0.8
    print(f"{c:8s} AUROC={roc_auc_score(df.em, df[c]):.3f} | approve@0.8={approve.mean():.2f} "
          f"| precision@0.8={df.em[approve].mean():.3f}")
print("n =", len(df))


In [ ]:
from datasets import load_dataset
hq = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation")
gold_ctx = {}
for ex in hq:
    keep = set(ex["supporting_facts"]["title"])
    gold_ctx[ex["question"]] = " ".join(" ".join(s) for t, s in zip(ex["context"]["title"], ex["context"]["sentences"]) if t in keep)
df["context"] = df["question"].map(gold_ctx).fillna(df["context"])
df["strands2"] = [score(r) for r in df.itertuples()]
print("strands (gold paragraphs) AUROC =", round(roc_auc_score(df.em, df.strands2), 3))
df.to_csv("jev_vs_strands.csv", index=False)


Below: New Oct 18

In [ ]:
from google.colab import files
up = files.upload()
print(list(up))

Saving jev_vs_strands.csv to jev_vs_strands.csv
['jev_vs_strands.csv']


In [ ]:
import pandas as pd
df = pd.read_csv("jev_vs_strands.csv")
print(df.columns.tolist()); print(len(df))

['id', 'question', 'draft', 'gold', 'em', 'jev', 'context', 'strands', 'strands2']
220


In [ ]:
!strands-decider serve --help | grep -i -E "len|ctx|token"
from transformers import AutoConfig
try: print(AutoConfig.from_pretrained("StrandsAgents/strands-decider-2B-hobson-v19").max_position_embeddings)
except Exception as e: print(e)

Unrecognized model in StrandsAgents/strands-decider-2B-hobson-v19. Should have a `model_type` key in its config.json.


In [ ]:
DRAFT = "draft"   # <-- change to your draft column name

from datasets import load_dataset
import torch, itertools, time
from types import SimpleNamespace
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import roc_auc_score

hq = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation")
need = set(df.question)
sub = hq.filter(lambda q: q in need, input_columns="question")
paras = {ex["question"]: [(t, f"{t}: {' '.join(s)}") for t, s in zip(ex["context"]["title"], ex["context"]["sentences"])] for ex in sub}
gold  = {ex["question"]: set(ex["supporting_facts"]["title"]) for ex in sub}
print("questions matched:", df.question.isin(paras).sum(), "/", len(df))

rn = "cross-encoder/ms-marco-MiniLM-L-6-v2"
rtok = AutoTokenizer.from_pretrained(rn)
rmod = AutoModelForSequenceClassification.from_pretrained(rn).cuda().eval()

@torch.no_grad()
def ranked(q, draft):
    ps = paras[q]
    x = rtok([f"{q} {draft}"] * len(ps), [p for _, p in ps], truncation=True, max_length=512, padding=True, return_tensors="pt").to("cuda")
    s = rmod(**x).logits.squeeze(-1).tolist()
    return [ps[i] for i in sorted(range(len(ps)), key=lambda i: -s[i])]

def S(r, ctx):  # calls Strands directly with the reranked evidence, no 3,000-char cut
    a = strands_critic(r.question, ctx, getattr(r, DRAFT))["answers"]["sufficient"]
    return a.get("noul", a.get("value"))

Filter:   0%|          | 0/7405 [00:00<?, ? examples/s]

questions matched: 220 / 220


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [ ]:
r = next(df.itertuples())
rk = ranked(r.question, getattr(r, DRAFT))
print([t for t, _ in rk[:3]], "| gold:", gold[r.question])
print("score:", S(r, "\n".join(p for _, p in rk[:3])))

['Carlos Alberto Hurtado', 'Carlos Morban', 'Carlos Alberto de Oliveira Júnior'] | gold: {'Club Necaxa', 'Carlos Alberto Hurtado'}
score: 0.2627


In [ ]:
import subprocess, time, requests
try: srv.kill()
except NameError: pass
log = open("server.log", "w")
srv = subprocess.Popen(["strands-decider", "serve", "StrandsAgents/strands-decider-2B-hobson-v19", "--port", "8000"], stdout=log, stderr=log)
for i in range(36):
    time.sleep(5)
    if srv.poll() is not None: print("server crashed"); break
    try: requests.get("http://localhost:8000", timeout=2); print("server up after", (i+1)*5, "s"); break
    except requests.exceptions.ConnectionError: pass
print(open("server.log").read()[-1500:])

server up after 20 s
serving StrandsAgents/strands-decider-2B-hobson-v19 on http://127.0.0.1:8000

Fetching 37 files: 100%|██████████| 37/37 [00:00<00:00, 1456.41it/s]

Loading weights: 100%|██████████| 320/320 [00:00<00:00, 9668.96it/s]
INFO:     Started server process [7489]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     127.0.0.1:58860 - "GET / HTTP/1.1" 404 Not Found



In [ ]:
from google.colab import drive; drive.mount("/content/drive")
out = "/content/drive/MyDrive/haltiq_strands_experiments.csv"
cols = {f"strands_top{k}": [] for k in (2,3,4)} | {"strands_pairmax": [], "recall_top3": [], "sec_top3": [], "sec_pairmax": []}

for i, r in enumerate(df.itertuples()):
    rk = ranked(r.question, getattr(r, DRAFT))
    for k in (2,3,4):
        t = time.time()
        cols[f"strands_top{k}"].append(S(r, "\n".join(p for _, p in rk[:k])))
        if k == 3: cols["sec_top3"].append(time.time() - t)
    cols["recall_top3"].append(gold[r.question] <= {t for t, _ in rk[:3]})
    t = time.time()
    cols["strands_pairmax"].append(max(S(r, f"{a}\n{b}") for (_, a), (_, b) in itertools.combinations(rk[:4], 2)))
    cols["sec_pairmax"].append(time.time() - t)
    if (i+1) % 20 == 0 or i+1 == len(df):
        part = df.iloc[:i+1].copy()
        for c, v in cols.items(): part[c] = v
        part.to_csv(out, index=False)
        print(f"{i+1}/{len(df)} saved", flush=True)

for c, v in cols.items(): df[c] = v
for c in ["strands_top2","strands_top3","strands_top4","strands_pairmax"]:
    print(c, "AUROC", round(roc_auc_score(df.em, df[c]), 3))
print("both-gold recall@3", round(df.recall_top3.mean(), 3))
print("sec/decision  top3", round(df.sec_top3.mean(), 2), " pairmax", round(df.sec_pairmax.mean(), 2))

Mounted at /content/drive
20/220 saved
40/220 saved
60/220 saved
80/220 saved
100/220 saved
120/220 saved
140/220 saved
160/220 saved
180/220 saved
200/220 saved
220/220 saved
strands_top2 AUROC 0.667
strands_top3 AUROC 0.623
strands_top4 AUROC 0.604
strands_pairmax AUROC 0.66
both-gold recall@3 0.75
sec/decision  top3 0.67  pairmax 3.13


In [ ]:
import numpy as np
rng = np.random.default_rng(0)
for c in ["strands","strands_top2","strands_top3","strands_top4","strands_pairmax","strands2","jev"]:
    if c not in df: continue
    y, s = df.em.values, df[c].values
    boots = []
    for _ in range(1000):
        i = rng.integers(0, len(y), len(y))
        if len(set(y[i])) == 2: boots.append(roc_auc_score(y[i], s[i]))
    appr = s >= 0.8
    prec = y[appr].mean() if appr.any() else float("nan")
    print(f"{c:16s} AUROC {roc_auc_score(y,s):.3f} [{np.percentile(boots,2.5):.3f}, {np.percentile(boots,97.5):.3f}]  approve@0.8 {appr.mean():.2f}  precision@0.8 {prec:.2f}  wrong-approvals {(appr & (y==0)).sum()}")
df.to_csv("/content/drive/MyDrive/haltiq_strands_experiments.csv", index=False)


strands          AUROC 0.434 [0.360, 0.508]  approve@0.8 0.07  precision@0.8 0.56  wrong-approvals 7
strands_top2     AUROC 0.667 [0.598, 0.732]  approve@0.8 0.47  precision@0.8 0.70  wrong-approvals 31
strands_top3     AUROC 0.623 [0.546, 0.697]  approve@0.8 0.34  precision@0.8 0.72  wrong-approvals 21
strands_top4     AUROC 0.604 [0.528, 0.675]  approve@0.8 0.35  precision@0.8 0.65  wrong-approvals 27
strands_pairmax  AUROC 0.660 [0.583, 0.732]  approve@0.8 0.61  precision@0.8 0.67  wrong-approvals 44
strands2         AUROC 0.734 [0.669, 0.803]  approve@0.8 0.61  precision@0.8 0.72  wrong-approvals 38
jev              AUROC 0.726 [0.655, 0.802]  approve@0.8 0.66  precision@0.8 0.71  wrong-approvals 42


In [ ]:
import json
bn = "BAAI/bge-reranker-base"
btok = AutoTokenizer.from_pretrained(bn)
bmod = AutoModelForSequenceClassification.from_pretrained(bn).cuda().eval()

@torch.no_grad()
def ranked_bge(q, draft):
    ps = paras[q]
    x = btok([f"{q} {draft}"] * len(ps), [p for _, p in ps], truncation=True, max_length=512, padding=True, return_tensors="pt").to("cuda")
    s = bmod(**x).logits.view(-1).float().tolist()
    return [ps[i] for i in sorted(range(len(ps)), key=lambda i: -s[i])]

out2 = "/content/drive/MyDrive/haltiq_strands_experiments2.csv"
cols2 = {k: [] for k in ["pair_scores", "recall_top2", "bge_top2", "bge_top3", "bge_recall_top2", "bge_recall_top3"]}

for i, r in enumerate(df.itertuples()):
    d = getattr(r, DRAFT)
    rk = ranked(r.question, d)
    cols2["recall_top2"].append(gold[r.question] <= {t for t, _ in rk[:2]})
    cols2["pair_scores"].append(json.dumps([S(r, f"{a}\n{b}") for (_, a), (_, b) in itertools.combinations(rk[:4], 2)]))
    bk = ranked_bge(r.question, d)
    for k in (2, 3):
        cols2[f"bge_top{k}"].append(S(r, "\n".join(p for _, p in bk[:k])))
        cols2[f"bge_recall_top{k}"].append(gold[r.question] <= {t for t, _ in bk[:k]})
    if (i+1) % 20 == 0 or i+1 == len(df):
        part = df.iloc[:i+1].copy()
        for c, v in cols2.items(): part[c] = v
        part.to_csv(out2, index=False)
        print(f"{i+1}/{len(df)} saved", flush=True)

for c, v in cols2.items(): df[c] = v
ps = df.pair_scores.map(json.loads)
df["pair_mean"] = ps.map(np.mean)
df["pair_2nd"] = ps.map(lambda x: sorted(x)[-2])
for c in ["strands_pairmax", "pair_mean", "pair_2nd", "strands_top2", "bge_top2", "strands_top3", "bge_top3"]:
    s = df[c].values; appr = s >= 0.8
    print(f"{c:16s} AUROC {roc_auc_score(df.em, s):.3f}  approve@0.8 {appr.mean():.2f}  wrong-approvals {(appr & (df.em==0)).sum()}")
print("MiniLM recall@2", round(df.recall_top2.mean(), 3), " recall@3", round(df.recall_top3.mean(), 3))
print("bge    recall@2", round(df.bge_recall_top2.mean(), 3), " recall@3", round(df.bge_recall_top3.mean(), 3))
df.to_csv(out2, index=False)

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

20/220 saved
40/220 saved
60/220 saved
80/220 saved
100/220 saved
120/220 saved
140/220 saved
160/220 saved
180/220 saved
200/220 saved
220/220 saved


IndexError: list index out of range

In [ ]:
df["pair_2nd"] = ps.map(lambda x: sorted(x)[-2] if len(x) > 1 else x[0])
print("rows with <6 pairs:", (ps.map(len) < 6).sum())
for c in ["strands_pairmax", "pair_mean", "pair_2nd", "strands_top2", "bge_top2", "strands_top3", "bge_top3"]:
    s = df[c].values; appr = s >= 0.8
    print(f"{c:16s} AUROC {roc_auc_score(df.em, s):.3f}  approve@0.8 {appr.mean():.2f}  wrong-approvals {(appr & (df.em==0)).sum()}")
print("MiniLM recall@2", round(df.recall_top2.mean(), 3), " recall@3", round(df.recall_top3.mean(), 3))
print("bge    recall@2", round(df.bge_recall_top2.mean(), 3), " recall@3", round(df.bge_recall_top3.mean(), 3))
df.to_csv(out2, index=False)

rows with <6 pairs: 2
strands_pairmax  AUROC 0.660  approve@0.8 0.61  wrong-approvals 44
pair_mean        AUROC 0.548  approve@0.8 0.06  wrong-approvals 4
pair_2nd         AUROC 0.599  approve@0.8 0.35  wrong-approvals 25
strands_top2     AUROC 0.667  approve@0.8 0.47  wrong-approvals 31
bge_top2         AUROC 0.725  approve@0.8 0.58  wrong-approvals 36
strands_top3     AUROC 0.623  approve@0.8 0.34  wrong-approvals 21
bge_top3         AUROC 0.659  approve@0.8 0.41  wrong-approvals 27
MiniLM recall@2 0.573  recall@3 0.75
bge    recall@2 0.836  recall@3 0.923


In [ ]:
y, s = df.em.values, df.bge_top2.values
b = [roc_auc_score(y[i], s[i]) for i in (rng.integers(0, len(y), len(y)) for _ in range(1000)) if len(set(y[i])) == 2]
print("bge_top2 AUROC CI", np.percentile(b, [2.5, 97.5]).round(3))


bge_top2 AUROC CI [0.648 0.793]
